# Layer Indexing

Walks the current Trace lookup semantics: exact labels, integer indexes, pass-qualified labels, suggestions, and missing labels.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_handle = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_handle.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(buffer_log_model, torch.randn(1, 4), layers_to_save="all")
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt)),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_handle,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Layer Indexing: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = []

exact = log["linear_1_1"]
ordinal = log[1]
qualified = log["linear_1_1:1"]
module_lookup = log["0:1"]
suggestions = log.find_layers("linar")
try:
    log["definitely_missing_layer"]
except Exception as exc:
    print(type(exc).__name__, str(exc).splitlines()[0][:100])
print(exact.layer_label, ordinal.layer_label, qualified.layer_label, module_lookup.layer_label)
inline_show("suggestions", suggestions[:3] if suggestions else suggestions)

## Explicit topical coverage

These names are also covered in anatomy notebooks, but this notebook owns the user-facing workflow.

In [ ]:
ITEMS = [
    "tl.Trace.layer_dict_all_keys",
    "tl.Trace.layer_dict_main_keys",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_list",
    "tl.Trace.layer_logs",
    "tl.Trace.suggest",
]

audit_touch_items("Explicit topical coverage", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.Trace.layer_dict_all_keys",
    "tl.Trace.layer_dict_main_keys",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_list",
    "tl.Trace.layer_logs",
    "tl.Trace.suggest",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")